# NGrams com corpus rDany

Aluno: Gabriel Melo. Matricula: 125.304-6

## Perguntas de Fechamento

* Que fração dos bigramas do teste não aparece no treino?

Após contagem, chegamos na fração 714/972 de bigramas não aparecem no conjunto de treino, isso é aproximadamente 73,46%.

* As sentenças geradas parecem português? Em que ponto elas desandam?

O rDany é em Inglês, logo o texto gerado é em algo que se assemelha ao Inglês (Vi alguns trechos com caracteres em russo). Quando olhamos o texto gerado com bigramas, conseguimos perceber coerência de até 3 palavras (apenas observação, sem um teste para suporte).

* O que acontece com a perplexidade ao passar de bigrama para trigrama? Por quê?

Ela aumenta, laplace penaliza mais pois o contexto é baixo nos trigramas e isso afeta negativamente o denominador, o que aumenta a perplexidade.

In [15]:
import pandas as pd
import numpy as np
import re
import math
import random
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

random.seed(42)

## Carregando o dataset

In [16]:
df = pd.read_csv('rdany-chat.csv', usecols=['source', 'text', 'date', 'hour'])
df.head()

,source,text,date,hour
0,human,[START],28/11/18,20:27:30
1,robot,Hi there,29/11/18,20:27:31
2,human,Oh,30/11/18,20:27:32
3,robot,here is afternoon!,01/12/18,20:27:33
4,human,How do you feel today? Tell me something about...,02/12/18,20:27:34


## Tokenização e Normalização

- Lowercase
- Remoção de URLs
- Remoção de emojis e caracteres especiais
- Tokenização por `\b\w+\b` (apenas palavras alfanuméricas)
- Inserção dos marcadores `<s>` (início) e `</s>` (fim)

In [17]:
def normalize_and_tokenize(text: str) -> list[str]:
    """Normaliza e tokeniza uma mensagem, retornando lista de tokens com marcadores."""
    if not isinstance(text, str):
        return []

    text = text.lower()
    # Remover URLs
    text = re.sub(r'https?://\S+', '', text)
    # Extrair apenas palavras alfanuméricas
    tokens = re.findall(r'\b\w+\b', text)

    if not tokens:
        return []

    return ['<s>'] + tokens + ['</s>']

# Aplicar a normalização e tokenização a cada mensagem
all_sentences = df['text'].apply(normalize_and_tokenize).tolist()
# Filtrar sentenças vazias
all_sentences = [s for s in all_sentences if len(s) > 2]

In [18]:
split = int(len(all_sentences) * 0.9)
train_sents = all_sentences[:split]
test_sents  = all_sentences[split:]

len(all_sentences), len(train_sents), len(test_sents)

(1702, 1531, 171)

## Contagem de Unigramas e Bigramas, método MLE

In [19]:
# --- Contagem ---
unigram_counts = Counter()
bigram_counts  = Counter()

for sent in train_sents:
    unigram_counts.update(sent)
    bigram_counts.update(zip(sent[:-1], sent[1:]))

V = len(unigram_counts)   # tamanho do vocabulário
N = sum(unigram_counts.values())  # total de tokens

print(f'Vocabulário (V):       {V} tipos')
print(f'Tokens totais (N):     {N}')
print(f'Bigramas distintos:    {len(bigram_counts)}')

Vocabulário (V):       1614 tipos
Tokens totais (N):     11222
Bigramas distintos:    5260


In [20]:
def p_mle(w_k: str, w_k1: str) -> float:
    """P(w_k | w_{k-1}) estimada por MLE."""
    denom = unigram_counts[w_k1]
    if denom == 0:
        return 0.0
    return bigram_counts[(w_k1, w_k)] / denom

# Teste
print(f'P("there" | "hi")  = {p_mle("there", "hi"):.4f}')
print(f'P("you"   | "are") = {p_mle("you", "are"):.4f}')
print(f'P("xyz"   | "the") = {p_mle("xyz", "the"):.4f}')

P("there" | "hi")  = 0.3030
P("you"   | "are") = 0.5217
P("xyz"   | "the") = 0.0000


## Método de Shannon

In [21]:
# Pré-computar as distribuições de transição para eficiência
transitions = defaultdict(list)   # w_{k-1} -> [(w_k, count), ...]
for (w_prev, w_next), count in bigram_counts.items():
    transitions[w_prev].append((w_next, count))

def generate_sentence(max_len: int = 50) -> str:
    """Gera uma sentença pelo método de amostragem de Shannon (bigrama)."""
    tokens = ['<s>']
    while len(tokens) < max_len:
        prev = tokens[-1]
        nexts = transitions.get(prev)
        if not nexts:
            break
        candidates, weights = zip(*nexts)
        chosen = random.choices(candidates, weights=weights, k=1)[0]
        if chosen == '</s>':
            break
        tokens.append(chosen)
    return ' '.join(tokens[1:])  # Remove <s>

print('=== 5 Sentenças Geradas (Shannon Sampling) ===\n')
for _ in range(5):
    print(f'{generate_sentence()}')

=== 5 Sentenças Geradas (Shannon Sampling) ===

she
i m home with you like to be nice
hi there how are you send them in this server
i really like mitsuku do
one love web


## Bigramas do Teste com Probabilidade Zero

Contamos quantos bigramas do conjunto de teste **nunca foram vistos** no treino, ou seja, $C(w_{k-1}, w_k) = 0$.

In [22]:
# Extrair todos os bigramas do teste
test_bigrams = []
for sent in test_sents:
    test_bigrams.extend(zip(sent[:-1], sent[1:]))

total_test_bg = len(test_bigrams)
zero_count    = sum(1 for bg in test_bigrams if bigram_counts[bg] == 0)

print(f'Total de bigramas no teste:            {total_test_bg}')
print(f'Bigramas com P_MLE = 0 (não vistos):   {zero_count}')
print(f'Proporção de zeros:                    {zero_count / total_test_bg:.2%}')

Total de bigramas no teste:            972
Bigramas com P_MLE = 0 (não vistos):   714
Proporção de zeros:                    73.46%


> **Observação:** a alta proporção de bigramas com probabilidade zero é esperada em corpora pequenos e vocabulário aberto (diálogos livres). Isso torna o modelo MLE puro inutilizável para calcular a probabilidade de sentenças do teste (qualquer sentença com um único bigrama zero recebe $P = 0$).

## Suavização de Laplace e Perplexidade

In [23]:
def p_laplace(w_k: str, w_k1: str) -> float:
    """P(w_k | w_{k-1}) com suavização de Laplace (Add-1)."""
    return (bigram_counts[(w_k1, w_k)] + 1) / (unigram_counts[w_k1] + V)

# Calcular log-probabilidade total no teste
log_prob_sum = 0.0
N_test = len(test_bigrams)

for (w_prev, w_next) in test_bigrams:
    prob = p_laplace(w_next, w_prev)
    log_prob_sum += math.log2(prob)

perplexity = 2 ** (-log_prob_sum / N_test)

print(f'Vocabulário V:               {V}')
print(f'Bigramas avaliados (N_test):  {N_test}')
print(f'Soma log2 P (Laplace):       {log_prob_sum:.2f}')
print(f'Perplexidade (PP):            {perplexity:.2f}')

Vocabulário V:               1614
Bigramas avaliados (N_test):  972
Soma log2 P (Laplace):       -9664.02
Perplexidade (PP):            983.93


## Trigramas

### Contar trigramas no treino

In [24]:
# Para trigramas, adicionamos um <s> extra no início de cada sentença
# para que o contexto de duas palavras exista desde o começo.

bigram_ctx_counts = Counter()   # C(w_{k-2}, w_{k-1}) — contexto de 2 palavras
trigram_counts    = Counter()   # C(w_{k-2}, w_{k-1}, w_k)

for sent in train_sents:
    padded = ['<s>'] + sent  # agora: <s> <s> w1 w2 ... </s>
    for i in range(len(padded) - 2):
        ctx = (padded[i], padded[i+1])
        tri = (padded[i], padded[i+1], padded[i+2])
        bigram_ctx_counts[ctx] += 1
        trigram_counts[tri]    += 1

print(f'Contextos de bigrama distintos: {len(bigram_ctx_counts)}')
print(f'Trigramas distintos:            {len(trigram_counts)}')

Contextos de bigrama distintos: 4567
Trigramas distintos:            6710


In [25]:
def p_mle_tri(w_k: str, w_k2: str, w_k1: str) -> float:
    """P(w_k | w_{k-2}, w_{k-1}) por MLE (trigrama)."""
    denom = bigram_ctx_counts[(w_k2, w_k1)]
    if denom == 0:
        return 0.0
    return trigram_counts[(w_k2, w_k1, w_k)] / denom

# Teste rápido
print(f'P("you" | "how", "are")   = {p_mle_tri("you", "how", "are"):.4f}')
print(f'P("there" | "<s>", "hi") = {p_mle_tri("there", "<s>", "hi"):.4f}')
print(f'P("xyz" | "the", "a")    = {p_mle_tri("xyz", "the", "a"):.4f}')

P("you" | "how", "are")   = 0.9714
P("there" | "<s>", "hi") = 0.3030
P("xyz" | "the", "a")    = 0.0000


### Geração de sentenças com trigramas (Shannon)

In [26]:
# Pré-computar transições de trigramas
tri_transitions = defaultdict(list)  # (w_{k-2}, w_{k-1}) -> [(w_k, count), ...]
for (w1, w2, w3), count in trigram_counts.items():
    tri_transitions[(w1, w2)].append((w3, count))

def generate_sentence_tri(max_len: int = 50) -> str:
    """Gera uma sentença pelo método de Shannon usando trigramas."""
    tokens = ['<s>', '<s>']
    while len(tokens) < max_len:
        ctx = (tokens[-2], tokens[-1])
        nexts = tri_transitions.get(ctx)
        if not nexts:
            break
        candidates, weights = zip(*nexts)
        chosen = random.choices(candidates, weights=weights, k=1)[0]
        if chosen == '</s>':
            break
        tokens.append(chosen)
    return ' '.join(t for t in tokens[2:])  # Remove os dois <s>

print('=== 5 Sentenças Geradas Trigramas (Shannon) ===\n')
for _ in range(5):
    print(f'{generate_sentence_tri()}')

=== 5 Sentenças Geradas Trigramas (Shannon) ===

juda ko p
so is this human being
is it
my boss
you can make a bot


### Trigramas do teste com probabilidade zero

In [27]:
# Extrair trigramas do teste (com padding duplo de <s>)
test_trigrams = []
for sent in test_sents:
    padded = ['<s>'] + sent
    for i in range(len(padded) - 2):
        test_trigrams.append((padded[i], padded[i+1], padded[i+2]))

total_test_tri  = len(test_trigrams)
zero_count_tri  = sum(1 for tri in test_trigrams if trigram_counts[tri] == 0)

print(f'Total de trigramas no teste:              {total_test_tri}')
print(f'Trigramas com P_MLE = 0 (não vistos):     {zero_count_tri}')
print(f'Proporção de zeros:                       {zero_count_tri / total_test_tri:.2%}')

Total de trigramas no teste:              972
Trigramas com P_MLE = 0 (não vistos):     757
Proporção de zeros:                       77.88%


### Perplexidade com Laplace (trigramas)

In [28]:
def p_laplace_tri(w_k: str, w_k2: str, w_k1: str) -> float:
    """P(w_k | w_{k-2}, w_{k-1}) com suavização de Laplace (trigramas)."""
    return (trigram_counts[(w_k2, w_k1, w_k)] + 1) / (bigram_ctx_counts[(w_k2, w_k1)] + V)

log_prob_sum_tri = 0.0
N_test_tri = len(test_trigrams)

for (w1, w2, w3) in test_trigrams:
    prob = p_laplace_tri(w3, w1, w2)
    log_prob_sum_tri += math.log2(prob)

perplexity_tri = 2 ** (-log_prob_sum_tri / N_test_tri)

print(f'Vocabulário V:                  {V}')
print(f'Trigramas avaliados (N_test):    {N_test_tri}')
print(f'Soma log2 P (Laplace):          {log_prob_sum_tri:.2f}')
print(f'Perplexidade (PP):               {perplexity_tri:.2f}')

Vocabulário V:                  1614
Trigramas avaliados (N_test):    972
Soma log2 P (Laplace):          -9813.25
Perplexidade (PP):               1094.41
